# GEDI 08B — Vertical Structure vs UAV Musanga (v2)

**Standalone 08B-only notebook.** This separate filename prevents the previously cached NB06 cell from being mistaken for the corrected version.

Run the code cell below. It uses Earth Engine project `musanga-511110` and cached Block 08A data. It dynamically reads available PAVD bands; it does not request `pavd_z30`.


## 08B — GEDI vertical structure versus Musanga (standalone; CPU; no training)

Uses the cached **08A matched 25 m GEDI raster locations** to sample quality-filtered **GEDI L2B monthly** observations (canopy cover, total PAI, foliage-height diversity, and PAVD across 31 height bins at 5 m steps). Filters on **`l2b_quality_flag=1` and `degrade_flag=0`**, which differs from L2A quality control. Supports sparse/missing L2B coverage and caches small batches of 125 points for safe Earth Engine requests. Aggregates valid raster locations to one record per frozen 50 m Musanga reference cell and reports **actual L2B-matched cell counts**.

The key scientific diagnostic is whether **FHD, PAI and vertical foliage profiles** relate more strongly to Musanga fraction and held-out S2+EMB residuals than RH98 height does. Tables include univariate correlations, descriptive in-sample ridge regressions of height vs structure vs combined, and per-forest figures. These are **exploratory relationships**, not LOFO generalization scores or a trained GEDI→Musanga predictor. Temporal matching to the UAV flight dates remains unresolved.

Run **08B only** after the successful 08A run. The established EE project `musanga-511110` is used automatically. Outputs: `CLEAN_NB06/08B_GEDI_vertical_structure/`.

Reference: [GEDI L2B monthly Earth Engine catalog](https://developers.google.com/earth-engine/datasets/catalog/LARSE_GEDI_GEDI02_B_002_MONTHLY).

In [ ]:
# 08B — GEDI L2B vertical structure vs UAV Musanga (standalone, CPU).
# Input: 08A matched 25m L2A raster-grid locations; no new spatial searches.
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from pyproj import Transformer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score,mean_absolute_error
import hashlib,json
import ee
try:
    ee.Initialize(project='musanga-511110')
except Exception:
    ee.Authenticate(auth_mode='colab')
    ee.Initialize(project='musanga-511110')
ROOTS08B=[
 Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06'),
 Path('/content/drive/MyDrive/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06')]
if not any((p/'08_GEDI_UAV_matching/08A_matched_GEDI_raster_pixels.parquet').is_file() for p in ROOTS08B):
    from google.colab import drive
    drive.mount('/content/drive')
available=[p for p in ROOTS08B if (p/'08_GEDI_UAV_matching/08A_matched_GEDI_raster_pixels.parquet').is_file()]
assert len(available)==1,('Run 08A first; no unambiguous GEDI-matched input found',available)
ROOT=available[0];IN=ROOT/'08_GEDI_UAV_matching'
OUT=ROOT/'08B_GEDI_vertical_structure';OUT.mkdir(parents=True,exist_ok=True)
pixels=pd.read_parquet(IN/'08A_matched_GEDI_raster_pixels.parquet').reset_index(drop=True)
cells=pd.read_parquet(IN/'08A_all_14454_cells.parquet').reset_index(drop=True)
assert {'site','gedi_x','gedi_y','reference_row'}.issubset(pixels)
assert {'site','mosaic','row','col','x','y','target_fraction','rh98_m','rh50_m'}.issubset(cells)
assert len(cells)==14454 and pixels.reference_row.ge(0).all()
pixels=pixels.sort_values(['site','reference_row','gedi_x','gedi_y']).reset_index(drop=True)
pixels['pixel_id']=np.arange(len(pixels),dtype=int)
# Query ACTUAL Earth Engine monthly L2B band inventory before building schema.
src=ee.ImageCollection('LARSE/GEDI/GEDI02_B_002_MONTHLY').filterDate(
    '2019-03-25','2026-01-01')
available_bands=src.first().bandNames().getInfo()
base=['cover','pai','fhd_normal']
quality_bands=['l2b_quality_flag','degrade_flag']
assert set(base+quality_bands).issubset(available_bands), (
    'Required L2B bands missing',sorted(set(base+quality_bands)-set(available_bands)))
pavd=sorted([b for b in available_bands if b.startswith('pavd_z') and
             b[len('pavd_z'):].isdigit()],key=lambda b:int(b[6:]))
assert len(pavd)>=10,('Unexpected L2B PAVD bands',pavd)
vars08B=base+pavd
print('Available L2B PAVD bands:',len(pavd),pavd[0],'through',pavd[-1],flush=True)
CFG={'version':'08B_L2B_monthly_structure_v2_dynamic_bands',
     'L2B':'LARSE/GEDI/GEDI02_B_002_MONTHLY',
     'date_range':['2019-03-25','2026-01-01'],
     'quality':'l2b_quality_flag==1 AND degrade_flag==0',
     'pixel_count':int(len(pixels)),'chunk_size':125,
     'source_hash':hashlib.sha256(pd.util.hash_pandas_object(
         pixels[['site','reference_row','gedi_x','gedi_y']],index=False
         ).values.tobytes()).hexdigest(),
     'features':vars08B,'method':'mean of quality-filtered monthly raster values sampled at 08A coordinates'}
config_path=OUT/'08B_v2_config.json'
if config_path.is_file():
    assert json.loads(config_path.read_text())==CFG,'Cached 08B inputs/config differ'
else: config_path.write_text(json.dumps(CFG,indent=2))
assert set(vars08B+quality_bands).issubset(available_bands),(
    'Missing L2B bands:',set(vars08B+quality_bands)-set(available_bands))
def quality08B(im):
    mask=im.select('l2b_quality_flag').eq(1).And(im.select('degrade_flag').eq(0))
    return im.select(vars08B).updateMask(mask)
valid08B=src.map(quality08B)
# Mean across available observation months; no chronological matching to UAV yet.
average08B=valid08B.mean().select(vars08B)
count08B=valid08B.select('cover').count().rename('n_L2B_monthly')
# -9999 is an explicit missing value; unmask avoids silently dropping the points.
sample_image08B=average08B.addBands(count08B).unmask(-9999)
to_lonlat=Transformer.from_crs('EPSG:32635','EPSG:4326',always_xy=True)
frames=[]
for lo in range(0,len(pixels),CFG['chunk_size']):
    hi=min(len(pixels),lo+CFG['chunk_size'])
    file=OUT/f'08B_v2_points_{lo:05d}_{hi:05d}.parquet'
    if file.is_file():
        piece=pd.read_parquet(file)
        print('CACHE 08B',hi,'/',len(pixels),flush=True)
    else:
        records=[]
        for row in pixels.iloc[lo:hi].itertuples(index=False):
            lon,lat=to_lonlat.transform(float(row.gedi_x),float(row.gedi_y))
            records.append(ee.Feature(ee.Geometry.Point([lon,lat]),
                                      {'pixel_id':int(row.pixel_id)}))
        fc=sample_image08B.sampleRegions(
            collection=ee.FeatureCollection(records),properties=['pixel_id'],
            scale=25,projection='EPSG:32635',geometries=False,tileScale=4)
        features=fc.getInfo()['features']
        piece=pd.DataFrame([f['properties'] for f in features])
        if not len(piece):piece=pd.DataFrame(columns=['pixel_id']+vars08B+['n_L2B_monthly'])
        assert piece.pixel_id.is_unique and set(piece.pixel_id).issubset(
            set(pixels.iloc[lo:hi].pixel_id))
        piece.to_parquet(file,index=False)
        print('EXTRACTED 08B',hi,'/',len(pixels),'L2B rows',len(piece),flush=True)
    frames.append(piece)
raw08B=pd.concat(frames,ignore_index=True)
raw08B=raw08B.drop_duplicates('pixel_id')
data08B=pixels.merge(raw08B,on='pixel_id',how='left',validate='one_to_one')
for col in vars08B+['n_L2B_monthly']:
    if col not in data08B.columns:data08B[col]=np.nan
    data08B[col]=pd.to_numeric(data08B[col],errors='coerce')
    data08B.loc[data08B[col]<=-9998,col]=np.nan
# L2B availability and quality are DIFFERENT from 08A L2A; do not assume all match.
data08B['has_L2B']=data08B.n_L2B_monthly.gt(0)&data08B[base].notna().all(axis=1)
data08B.loc[~data08B.has_L2B,vars08B]=np.nan
# Vertical PAVD is nonnegative and measured at 5m height steps. Aggregate into
# biologically interpretable LOW 0-10m, MID 15-25m, HIGH 30+m summaries.
for label,idx in [('low_0_10',[0,1,2]),('mid_15_25',[3,4,5]),('high_30plus',[int(q[6:]) for q in pavd if int(q[6:])>=6])]:
    names=[f'pavd_z{i}' for i in idx]
    data08B[f'pavd_{label}']=data08B[names].clip(lower=0).mean(axis=1)
# Vertical distribution fractions, defined only where positive PAVD is available.
parts=['pavd_low_0_10','pavd_mid_15_25','pavd_high_30plus']
data08B['pavd_total_summary']=data08B[parts].sum(axis=1,min_count=3)
data08B['pavd_mid_fraction']=np.where(data08B.pavd_total_summary>0,
                   data08B.pavd_mid_15_25/data08B.pavd_total_summary,np.nan)
data08B['pavd_high_fraction']=np.where(data08B.pavd_total_summary>0,
                   data08B.pavd_high_30plus/data08B.pavd_total_summary,np.nan)
# One row per reference cell: aggregation across matched raster locations.
FEATURES08B=['cover','pai','fhd_normal','pavd_low_0_10',
             'pavd_mid_15_25','pavd_high_30plus',
             'pavd_mid_fraction','pavd_high_fraction']
valid_pixels08B=data08B.loc[data08B.has_L2B].copy()
if len(valid_pixels08B):
     grouped08B=valid_pixels08B.groupby('reference_row').agg(
        **{v:(v,'mean') for v in FEATURES08B},
        n_L2B_raster_pixels=('pixel_id','size'),
        n_L2B_monthly_observations=('n_L2B_monthly','sum'))
else:
    grouped08B=pd.DataFrame(columns=FEATURES08B+['n_L2B_raster_pixels','n_L2B_monthly_observations'])
    grouped08B.index.name='reference_row'
joined08B=cells.join(grouped08B,how='left')
joined08B['has_L2B']=joined08B.n_L2B_raster_pixels.notna()
joined08B['musanga_pct']=joined08B.target_fraction*100
matched08B=joined08B[joined08B.has_L2B].copy()
coverage08B=(joined08B.groupby('site').agg(
    n_reference=('has_L2B','size'),n_L2B_matched=('has_L2B','sum')))
coverage08B['match_pct']=100*coverage08B.n_L2B_matched/coverage08B.n_reference
print('08B L2B coverage INSIDE original 50m reference cells:')
display(coverage08B.round(2))
coverage08B.to_csv(OUT/'08B_coverage.csv')
data08B.to_parquet(OUT/'08B_L2B_raster_locations.parquet',index=False)
joined08B.to_parquet(OUT/'08B_all_frozen_cells.parquet',index=False)
# Univariate correlations and partial R² beyond rh98, using separate regressions.
def fit_r2(y,x):
    ok=np.isfinite(y)&np.isfinite(x).all(axis=1)
    if ok.sum()<12:return np.nan,int(ok.sum())
    xx=x[ok];yy=y[ok]
    if np.any(np.std(xx,axis=0)<1e-8):return np.nan,int(ok.sum())
    model=make_pipeline(StandardScaler(),Ridge(alpha=1.0))
    model.fit(xx,yy)
    return float(r2_score(yy,model.predict(xx))),int(ok.sum())
rows=[]
for forest,frame in [('POOLED (descriptive)',matched08B)]+list(matched08B.groupby('site')):
    for v in ['rh98_m','rh50_m']+FEATURES08B:
        sub=frame[['musanga_pct',v]].dropna()
        if len(sub)<12 or sub[v].std()<=1e-8:continue
        rr=float(sub.musanga_pct.corr(sub[v]))
        rows.append(dict(forest=forest,feature=v,n=len(sub),r=rr,univariate_R2=rr**2,
                         slope_pp_per_unit=float(np.polyfit(sub[v],sub.musanga_pct,1)[0])))
stats08B=pd.DataFrame(rows)
print('GEDI 08B descriptive STRUCTURE correlations versus UAV Musanga:')
display(stats08B.round(3))
stats08B.to_csv(OUT/'08B_structure_vs_Musanga_univariate.csv',index=False)
combined=[]
for forest,group in [('POOLED (descriptive)',matched08B)]+list(matched08B.groupby('site')):
    needed=['musanga_pct','rh98_m']+FEATURES08B
    tab=group[needed].replace([np.inf,-np.inf],np.nan).dropna()
    if len(tab)<25:continue
    y=tab.musanga_pct.to_numpy(float)
    height_r2,_=fit_r2(y,tab[['rh98_m']].to_numpy(float))
    struct_r2,_=fit_r2(y,tab[FEATURES08B].to_numpy(float))
    both_r2,_=fit_r2(y,tab[['rh98_m']+FEATURES08B].to_numpy(float))
    combined.append(dict(forest=forest,n=len(tab),height_R2_in_sample=height_r2,
        structure_R2_in_sample=struct_r2,combined_R2_in_sample=both_r2,
        delta_R2_beyond_height=both_r2-height_r2))
combined08B=pd.DataFrame(combined)
print('Exploratory MULTIVARIATE association (in-sample; not held-out validation):')
display(combined08B.round(3))
combined08B.to_csv(OUT/'08B_structural_increment_in_sample.csv',index=False)
# Extra check: same matched subset of S2 LOFO predictions, never tune on them.
pred_file=ROOT/'04_matched_50m_planet'/'04B_exact_matched_predictions_50m.parquet'
if pred_file.is_file() and len(matched08B):
    keys=['site','mosaic','row','col']
    pred=pd.read_parquet(pred_file,columns=keys+['reference_pct','S2_EMB_predicted_pct'])
    pred['S2_residual_pp']=pred.S2_EMB_predicted_pct-pred.reference_pct
    residual=matched08B.merge(pred[keys+['S2_residual_pp']],on=keys,how='inner',
                              validate='one_to_one')
    residual_rows=[]
    for name,g in [('POOLED',residual)]+list(residual.groupby('site')):
        for feat in ['rh98_m']+FEATURES08B:
            t=g[[feat,'S2_residual_pp']].dropna()
            if len(t)>=12 and t[feat].std()>1e-8:
                residual_rows.append(dict(forest=name,feature=feat,n=len(t),
                    r_with_S2_error=t[feat].corr(t.S2_residual_pp)))
    residual_stats=pd.DataFrame(residual_rows)
    print('L2B structure versus strict LOFO S2 prediction errors:')
    display(residual_stats.round(3))
    residual_stats.to_csv(OUT/'08B_structure_vs_S2_error.csv',index=False)
    residual.to_parquet(OUT/'08B_matched_with_S2_LOFO_residual.parquet',index=False)
# Figures per forest, each GEDI-matched 50m reference cell plotted once.
fig,axes=plt.subplots(2,2,figsize=(11,9),layout='constrained')
for ax,site in zip(axes.flat,['Baego','MOS','Yangambi','Yoko']):
    g=matched08B.loc[matched08B.site==site]
    if len(g):
        ax.scatter(g.fhd_normal,g.musanga_pct,alpha=.5,s=12)
    ax.set(xlabel='GEDI Foliage Height Diversity',ylabel='UAV Musanga cover (%)',
           title=f'{site}: n={len(g)}',ylim=(-2,102))
    ax.grid(alpha=.15)
fig.savefig(OUT/'08B_FHD_vs_Musanga.png',dpi=180)
plt.show();plt.close(fig)
print('08B COMPLETE, outputs:',OUT)
print('Do NOT treat L2A and L2B monthly raster values as simultaneous laser shots.')
print('Chronology, footprint geometry and 50m-support differences remain unresolved.')
